# 🗂️ Subset & Model Selection
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, which ones should you include in the model? Subset selection methods systematically compare model subsets. Information criteria (Cp, BIC, adjusted R²) give principled ways to choose model size without a test set.

### What you'll learn
- Best subset selection: exhaustive search
- Forward and backward stepwise selection: greedy approximations
- Cp, BIC, and adjusted R² as model selection criteria
- Practical comparison with cross-validation

### Dataset: Hitters (ISLP) — 19 baseball predictors
### Time: ~50 minutes

## 🎯 Part 1 — Why Variable Selection?

With p predictors, using all of them can overfit — especially when p is large relative to n. Including irrelevant predictors:
- Increases variance of estimates
- Reduces interpretability
- May worsen prediction on new data

We want the smallest model that explains the data well.

In [ ]:
# Forward stepwise selection (greedy — each step adds best remaining feature)
def forward_stepwise(X, y, feature_names, max_features=None):
    if max_features is None: max_features = X.shape[1]
    selected = []
    remaining = list(range(X.shape[1]))
    results = []
    lr = LinearRegression()
    for step in range(max_features):
        best_score, best_feat = -np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            cv = cross_val_score(lr, X[:, candidate], y, cv=5,
                                scoring='neg_mean_squared_error').mean()
            if cv > best_score:
                best_score, best_feat = cv, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        results.append({'n_features': step+1, 'features': selected.copy(),
                       'cv_mse': -best_score,
                       'last_added': feature_names[best_feat]})
    return pd.DataFrame(results)

print('Running forward stepwise selection...')
fwd_results = forward_stepwise(X, y, features, max_features=15)
print(fwd_results[['n_features','last_added','cv_mse']].to_string())
print(f'\nBest model size: {fwd_results["cv_mse"].idxmin()+1} features')

In [ ]:
# Plot: CV MSE vs number of features
fig, ax = plt.subplots(figsize=(10, 4))
best_n = fwd_results['cv_mse'].idxmin()
ax.plot(fwd_results['n_features'], fwd_results['cv_mse'], 'o-', color='#1e5fa8', lw=2, markersize=6)
ax.axvline(fwd_results.loc[best_n, 'n_features'], color='#e85d20', lw=1.5, ls='--',
          label=f'Best: {fwd_results.loc[best_n,"n_features"]} features (CV MSE={fwd_results.loc[best_n,"cv_mse"]:.4f})')
ax.set_xlabel('Number of features')
ax.set_ylabel('5-fold CV MSE (log salary)')
ax.set_title('Forward Stepwise Selection — Hitters Dataset')
ax.legend()
plt.tight_layout(); plt.show()
best_features = fwd_results.loc[best_n, 'features']
print(f'\nSelected features: {[features[i] for i in best_features]}')

In [ ]:
# Information criteria: AIC, BIC, adjusted R²
# (computed on full training data — no cross-validation needed)
import statsmodels.api as sm

n = len(y)
X_sm = sm.add_constant(X[:, :10])  # first 10 features for illustration
model_sm = sm.OLS(y, X_sm).fit()

print('Model selection criteria (statsmodels):')  
print(f'  AIC:  {model_sm.aic:.2f}')
print(f'  BIC:  {model_sm.bic:.2f}')
print(f'  R²:   {model_sm.rsquared:.4f}')
print(f'  Adj R²: {model_sm.rsquared_adj:.4f}')
print()
print('Building models of increasing size and comparing criteria...')
for k in [1, 3, 5, 8, 10, 12, 15]:
    if k <= len(features):
        Xk = sm.add_constant(X[:, :k])
        mk = sm.OLS(y, Xk).fit()
        print(f'  p={k:<3} AIC={mk.aic:.1f}  BIC={mk.bic:.1f}  AdjR²={mk.rsquared_adj:.4f}')

In [ ]:
# Exercise workspace
# Task 1: Implement backward stepwise selection (start with all features, remove worst one each step)
# YOUR CODE HERE

# Task 2: Compare forward stepwise vs Lasso (from regularization notebook) on Hitters
# Which selects fewer features? Which has lower CV MSE?
from sklearn.linear_model import LassoCV
# YOUR CODE HERE

# Task 3: What features does the best 5-feature model select? Interpret each one
best_5 = fwd_results.loc[fwd_results['n_features']==5, 'features'].values[0]
print('Top 5 features:', [features[i] for i in best_5])
# YOUR CODE HERE — fit and report coefficients

In [ ]:
# @title 📝 Quiz — Subset & Model Selection
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is best subset selection and why is it computationally expensive?
# @markdown **Q2:** What is the difference between forward and backward stepwise selection?
# @markdown **Q3:** What does BIC penalize compared to AIC?
# @markdown **Q4:** Why is adjusted R² better than R² for model comparison?
# @markdown **Q5:** When would you use stepwise selection vs Lasso for variable selection?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_="Subset & Model Selection"
# @title 🤖 AI Grading
# @markdown **Enter your GitHub username, then click ▶ Run.**
# @markdown Colab will send your answers to Gemini automatically — no key needed.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

import textwrap
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {v or '(no answer)'}"
                    for i,(_,v) in enumerate(answers.items(),1))

    prompt = (f"Grade these quiz answers for the \"{_NB_TITLE}\" notebook "
              f"(Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
              f"{qa}\n\n"
              f"For each answer say CORRECT, PARTIAL, or INCORRECT and one sentence why. "
              f"Accept any reasonable paraphrase as correct. "
              f"End with an overall grade (Excellent/Good/Needs Review/Incomplete) "
              f"and one sentence on what to review if they struggled.")

    # Try Colab's built-in Gemini via the generativeai SDK (no key needed in Colab)
    success = False
    try:
        import google.generativeai as genai
        # In Colab, calling generate_content without configure() uses
        # Colab's own managed credentials automatically
        model = genai.GenerativeModel("gemini-1.5-flash")
        resp  = model.generate_content(prompt)
        print("\n" + "\u2550"*56)
        print(f"  \U0001f916 Feedback \u2014 {_NB_TITLE}")
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}")
        print("\u2550"*56)
        for line in resp.text.strip().split("\n"):
            if line.strip():
                for w in textwrap.wrap(line.strip(), 54):
                    print(f"  {w}")
            else:
                print()
        print("\u2550"*56)
        success = True
    except Exception:
        pass

    if not success:
        # Fallback: print the prompt so they can paste it into Colab's AI chat
        print("\u2550"*56)
        print("  Automatic grading unavailable.")
        print("  Paste the text below into the Gemini chat panel:")
        print("  (click the \u2728 sparkle icon in the Colab toolbar)")
        print("\u2550"*56)
        print()
        print(prompt)
        print()
        print("\u2550"*56)

    print("\n  \U0001f4be File \u2192 Save a copy in GitHub \u2192 your fork")
